# 🔥 Wildfire Decision Support — Pipeline (Colab)
Run cells **in order**. Data is saved to Google Drive so you can resume after a disconnect.

**Steps:**
1. Mount Drive & install deps
2. Fill in your credentials (CDS key, GitHub URL)
3. Run the pipeline — progress is checkpointed automatically
4. Sync `data/` to your local machine, then run `python import_colab_manifest.py`

## Cell 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 · Install dependencies

In [ ]:
# Install all dependencies (matches requirements.txt)
!pip install -q \
    flask flask-cors flask-sqlalchemy PyJWT python-dotenv \
    rasterio numpy scikit-learn xgboost geopandas pyarrow \
    geoalchemy2 cdsapi requests xarray netCDF4 dask \
    shapely pyproj pandas tqdm python-dateutil anthropic \
    'git+https://github.com/RayPan-UC/wildfire-hotspot-prediction.git'

print('✓ Dependencies installed')

## Cell 3 · Credentials & paths
Get your CDS API key from https://cds.climate.copernicus.eu/profile

In [ ]:
import os, pathlib

# ── Fill these in ──────────────────────────────────────────────────────────────
GITHUB_TOKEN = 'YOUR-GITHUB-PAT'                                          # <── GitHub personal access token (repo scope)
CDS_API_KEY  = 'ba4f70c2-cced-4275-b60d-a9e373562f3a'                    # <── your CDS key
# ──────────────────────────────────────────────────────────────────────────────

# Build authenticated clone URL (token is never printed)
GITHUB_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/RayPan-UC/wildfire-decision-support'

# Write CDS API config
cdsrc = pathlib.Path.home() / '.cdsapirc'
if not cdsrc.exists():
    cdsrc.write_text(f'url: https://cds.climate.copernicus.eu/api/v2\nkey: {CDS_API_KEY}\n')
    print('✓ ~/.cdsapirc written')
else:
    print('✓ ~/.cdsapirc already exists')

## Cell 4 · Clone repo & set up paths

In [ ]:
import subprocess, sys, pathlib

# ── Set this to the Google Drive folder where you uploaded data/ ──────────────
# e.g. if you uploaded into MyDrive/wf_data/, set:
#   DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/wf_data')
DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/WDS')
# ─────────────────────────────────────────────────────────────────────────────

DRIVE_DATA.mkdir(parents=True, exist_ok=True)

# Clone / update repo
REPO_DIR = pathlib.Path('/content/wildfire-decision-support')
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth=1', GITHUB_REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull'])

# Add backend to Python path
BACKEND = REPO_DIR / 'backend'
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

# Symlink repo/data/ → Drive so output persists across restarts
DATA_DIR = REPO_DIR / 'data'
if not DATA_DIR.exists():
    DATA_DIR.symlink_to(DRIVE_DATA)
    print(f'✓ data/ → {DRIVE_DATA}')
else:
    print(f'✓ data/ exists at {DATA_DIR.resolve()}')

STATE_PATH = DATA_DIR / 'colab_state.json'
print(f'Repo: {REPO_DIR}  |  Data: {DATA_DIR}')

## Cell 5 · Event config & mock DB layer

In [ ]:
import json
from datetime import date
from dataclasses import dataclass
from typing import Optional
from unittest.mock import MagicMock
from shapely.geometry import box as _shapely_box
from shapely import wkb as _shapely_wkb

EVENT_CONFIG = {
    'id':         1,
    'name':       'Fort McMurray Wildfire 2016',
    'year':       2016,
    'bbox':       (-112.634, 56.157, -110.002, 57.380),
    'start_date': '2016-05-01',
    'end_date':   '2016-05-31',
}

@dataclass
class _MockBBox:
    data: bytes
    @classmethod
    def from_bounds(cls, lon_min, lat_min, lon_max, lat_max):
        return cls(data=_shapely_wkb.dumps(_shapely_box(lon_min, lat_min, lon_max, lat_max)))

@dataclass
class _MockEvent:
    id: int; name: str; year: int
    bbox: _MockBBox; start_date: date; end_date: Optional[date]

_MOCK_EVENT = _MockEvent(
    id         = EVENT_CONFIG['id'],
    name       = EVENT_CONFIG['name'],
    year       = EVENT_CONFIG['year'],
    bbox       = _MockBBox.from_bounds(*EVENT_CONFIG['bbox']),
    start_date = date.fromisoformat(EVENT_CONFIG['start_date']),
    end_date   = date.fromisoformat(EVENT_CONFIG['end_date']),
)

_geo = MagicMock()
_geo.Geometry = lambda *a, **kw: None
import sys as _sys
_sys.modules.setdefault('geoalchemy2',          _geo)
_sys.modules.setdefault('geoalchemy2.types',    _geo)
_sys.modules.setdefault('geoalchemy2.elements', _geo)

_models_mock = MagicMock()
_models_mock.FireEvent.query.get.return_value = _MOCK_EVENT
_sys.modules['db.models']     = _models_mock
_sys.modules['db.connection'] = MagicMock()
_sys.modules['db']            = MagicMock()

print('✓ Event config and DB mock ready')

## Cell 6 · Prepare environment
**If you already uploaded `data/` from your local machine, all steps below will print `skip` and finish in seconds.**

In [ ]:
import wildfire_hotspot_prediction as whp

MODELS_DIR = DATA_DIR / 'static' / 'models'

print('[1/6] ML models …')
whp.ensure_models(models_dir=MODELS_DIR)

lon_min, lat_min, lon_max, lat_max = EVENT_CONFIG['bbox']
project_dir = DATA_DIR / 'events' / f"{EVENT_CONFIG['year']}_{EVENT_CONFIG['id']:04d}"
study = whp.Study(
    name        = EVENT_CONFIG['name'],
    bbox        = (lon_min, lat_min, lon_max, lat_max),
    start_date  = EVENT_CONFIG['start_date'],
    end_date    = EVENT_CONFIG['end_date'],
    project_dir = project_dir,
)
study.makedirs()
for _u in (study.models_dir, study.predictions_dir, study.data_render_dir):
    try: _u.rmdir()
    except OSError: pass

print('[2/6] ERA5 …')
whp.ensure_era5_coverage(study)

if not (study.landcover_raw_dir / 'fuel_type.tif').exists():
    print('[3/6] Downloading landcover …')
    whp.collect_environment(study, sources=['landcover'])
if not (study.landcover_dir / 'fuel_type.tif').exists():
    whp.preprocess_environment(study, sources=['landcover'])
print('[3/6] Landcover ✓')

import rasterio; from rasterio.crs import CRS
_WKT_3978 = ('PROJCS["NAD83 / Canada Atlas Lambert",GEOGCS["NAD83",DATUM["North_American_Datum_1983",'
             'SPHEROID["GRS 1980",6378137,298.257222101]],PRIMEM["Greenwich",0],'
             'UNIT["degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],'
             'PARAMETER["standard_parallel_1",49],PARAMETER["standard_parallel_2",77],'
             'PARAMETER["latitude_of_origin",49],PARAMETER["central_meridian",-95],'
             'PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1]]')
_crs3978 = CRS.from_wkt(_WKT_3978)
terrain_raw = study.project_dir / 'data_raw' / 'terrain'
if not (terrain_raw / 'dtm.tif').exists():
    print('[4/6] Downloading terrain …')
    whp.collect_environment(study, sources=['terrain'])
for _fname in ('dtm.tif', 'slope.tif', 'aspect.tif'):
    _p = terrain_raw / _fname
    if not _p.exists(): continue
    with rasterio.open(_p) as _src:
        try:
            if _src.crs and _src.crs.to_epsg() == 3978: continue
        except Exception: pass
        _data = _src.read(); _meta = _src.meta.copy()
    _meta['crs'] = _crs3978
    _tmp = _p.with_suffix('.patched.tif')
    with rasterio.open(_tmp, 'w', **_meta) as _dst: _dst.write(_data)
    _tmp.replace(_p)
if not (study.project_dir / 'data_processed' / 'terrain' / 'dtm.tif').exists():
    whp.preprocess_environment(study, sources=['terrain'])
print('[4/6] Terrain ✓')

if not (study.weather_dir / 'ffmc_daily.parquet').exists():
    print('[5/6] Building FWI …')
    whp.build_fire_weather_index(study)
print('[5/6] FWI ✓')

if not (study.data_processed_dir / 'grid_static.parquet').exists():
    print('[6/6] Building static grid …')
    whp.build_grid(study)
print('[6/6] Grid ✓')

fire_state_path = study.data_processed_dir / 'training' / 'fire_state.pkl'
if not fire_state_path.exists():
    print('[+] FIRMS hotspots + fire_state (this may take a while) …')
    whp.collect_hotspots(study)
    hotspot_data = whp.preprocess_hotspots(study)
    from wildfire_hotspot_prediction.training.fire_state import build_fire_state, save_fire_state
    save_fire_state(build_fire_state(hotspot_data), fire_state_path)
print(f'fire_state ✓  →  {fire_state_path}')
print('\n✅ Environment ready — proceeding to pipeline')

## Cell 7 · Run pipeline
Predict + spatial + weather for every timestep. Checkpointed — safe to interrupt and re-run.

In [ ]:
import json, re
import pandas as pd
from datetime import timedelta
from tqdm import tqdm
from wildfire_hotspot_prediction.training.fire_state import load_fire_state
from pipeline.predict import run_prediction
from pipeline.spatial import run_spatial_analysis
from pipeline.weather import build_weather_forecast
from pipeline.predict.risk_zones import load_youden_threshold

fire_state = load_fire_state(fire_state_path)
predictor  = whp.WildfirePredictor(models_dir=MODELS_DIR, model_name='lr_steps')
threshold  = load_youden_threshold(MODELS_DIR)
pred_cache = whp.build_prediction_cache(study)

state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}

def _save_state(s):
    STATE_PATH.write_text(json.dumps(s, indent=2, default=str))

def _ts_dir(slot):
    ts_str = pd.Timestamp(slot).strftime('%Y-%m-%dT%H%M')
    return DATA_DIR / 'events' / f"{EVENT_CONFIG['year']}_{EVENT_CONFIG['id']:04d}" / 'timesteps' / ts_str

def _merge_roads(spat_dir, road_summary):
    ctx = spat_dir.parent / 'prediction' / 'fire_context.json'
    if not ctx.exists(): return
    try:
        d = json.loads(ctx.read_text())
        d['road_summary'] = road_summary
        ctx.write_text(re.sub(r'\bNaN\b', 'null', json.dumps(d, ensure_ascii=False, indent=2)))
    except Exception as e:
        print(f'WARN road merge: {e}')

steps = sorted(fire_state.steps)
start = pd.Timestamp(EVENT_CONFIG['start_date']).floor('3h')
end   = pd.Timestamp(EVENT_CONFIG['end_date']) + pd.Timedelta(hours=23, minutes=59)
t, slots = start, []
while t <= end:
    slots.append(t); t += timedelta(hours=3)

_GAP_H = 12.0
slot_pairs = []
for slot in slots:
    past = [s for s in steps if s <= slot]
    if not past: continue
    t1 = max(past)
    slot_pairs.append((slot, t1, (slot - t1).total_seconds() / 3600.0))

print(f'Total timesteps: {len(slot_pairs)}')
already = sum(1 for v in state.values() if v.get('prediction_status') == 'done')
print(f'Already done (checkpoint): {already}')

with tqdm(slot_pairs, desc='Pipeline', unit='ts', dynamic_ncols=True) as pbar:
    for slot, t1, gap_h in pbar:
        key = pd.Timestamp(slot).strftime('%Y-%m-%dT%H:%M')
        ts  = state.setdefault(key, {
            'slot_time': slot.isoformat(), 'nearest_t1': t1.isoformat(),
            'gap_hours': round(gap_h, 2),  'data_gap_warn': gap_h > _GAP_H,
            'prediction_status': 'pending', 'spatial_analysis_status': 'pending',
            'affected_population': None, 'at_risk_3h': None,
            'at_risk_6h': None, 'at_risk_12h': None,
        })
        ts_dir   = _ts_dir(slot)
        pred_dir = ts_dir / 'prediction'
        spat_dir = ts_dir / 'spatial_analysis'
        wx_dir   = ts_dir / 'weather'
        t1n      = t1.tz_localize(None) if t1.tzinfo else t1

        if ts['prediction_status'] not in ('done', 'failed'):
            pbar.set_postfix_str(f'{key[:10]} predict')
            try:
                run_prediction(ts_id=0, overpass_time=t1n, study=study,
                               fire_state=fire_state, predictor=predictor,
                               threshold=threshold, out_dir=pred_dir, pred_cache=pred_cache)
                ts['prediction_status'] = 'done'
            except Exception as e:
                tqdm.write(f'WARN predict {key}: {e}')
                ts['prediction_status'] = 'failed'
            _save_state(state)

        if ts['prediction_status'] == 'done' and ts['spatial_analysis_status'] not in ('done', 'failed'):
            pbar.set_postfix_str(f'{key[:10]} spatial')
            try:
                counts = run_spatial_analysis(EVENT_CONFIG['id'], 0, spat_dir)
                if counts:
                    rs = counts.pop('road_summary', None)
                    if rs: _merge_roads(spat_dir, rs)
                    for k in ('affected_population', 'at_risk_3h', 'at_risk_6h', 'at_risk_12h'):
                        if k in counts: ts[k] = counts[k]
                ts['spatial_analysis_status'] = 'done'
            except Exception as e:
                tqdm.write(f'WARN spatial {key}: {e}')
                ts['spatial_analysis_status'] = 'failed'
            _save_state(state)

        if not (wx_dir / 'forecast.json').exists():
            pbar.set_postfix_str(f'{key[:10]} weather')
            try:
                build_weather_forecast(study=study, t1=t1n, out_dir=wx_dir)
            except Exception as e:
                tqdm.write(f'WARN weather {key}: {e}')

done = sum(1 for v in state.values() if v.get('prediction_status') == 'done')
fail = sum(1 for v in state.values() if v.get('prediction_status') == 'failed')
print(f'\n✅ Done: {done} succeeded, {fail} failed')

## Cell 8 · Export manifest for local DB import

In [ ]:
manifest = {
    'event': EVENT_CONFIG,
    'timesteps': [{**v, 'event_id': EVENT_CONFIG['id']} for v in state.values()],
}
out = DATA_DIR / 'colab_manifest.json'
out.write_text(json.dumps(manifest, indent=2, default=str))
print(f'✅ Manifest → {out}  ({len(manifest["timesteps"])} timesteps)')
print('\nNext: sync Drive data/ to local, then run  python import_colab_manifest.py')